In [47]:
import sympy as sp
import os
import re

### State space model

In [48]:
frequency = 100  # Hz
dt = 1.0/frequency

# this model use a state space model:
# x_k = F * x_{k-1} + B * u_k + w_k
# y_k = H * x_k + v_k

# 1. defines the state transition matrix (F)
F = sp.Matrix([
    [1, dt],
    [0,  1]
])

# 2. defines the control input matrix (B)
B = sp.Matrix([
    [0.5 * dt**2],
    [dt         ]
])

# 3. defines the measurement matrix (H)
H = sp.Matrix([
    [1, 0]
])

### Get the dimensions and formulate the matrices.

In [49]:
# get the dimensions of the matrices
dim_x = F.shape[0]
dim_u = B.shape[1] if B else 0
dim_z = H.shape[0]

def create_sym_matrix(name, rows, cols, is_vector=False):
    if is_vector:
        return sp.Matrix(rows, 1, lambda i, j: sp.Symbol(f'{name}[{i}]'))
    else:
        return sp.Matrix(rows, cols, lambda i, j: sp.Symbol(f'{name}[{i}][{j}]'))

x = create_sym_matrix('x', dim_x, 1, is_vector=True)
u = create_sym_matrix('u', dim_u, 1, is_vector=True) if dim_u > 0 else None
z = create_sym_matrix('z', dim_z, 1, is_vector=True)

P = create_sym_matrix('P', dim_x, dim_x)
Q = create_sym_matrix('Q', dim_x, dim_x)
R = create_sym_matrix('R', dim_z, dim_z)
I = sp.eye(dim_x)

print(f"#define LKF_STATE_DIM {dim_x}")
print(f"#define LKF_MEASURE_DIM {dim_z}")
print(f"#define LKF_CONTROL_DIM {dim_u}")

#define LKF_STATE_DIM 2
#define LKF_MEASURE_DIM 1
#define LKF_CONTROL_DIM 1


### Print the c++ code

In [50]:
# update the Kalman Filter equations symbolically
def print_cpp_block(expr_matrix, prefix):
    print(f"    // --- calculating {prefix} ---")
    for i in range(expr_matrix.rows):
        for j in range(expr_matrix.cols):
            val = sp.ccode(sp.simplify(expr_matrix[i, j]))
            if expr_matrix.cols == 1:
                print(f"    float next_{prefix}_{i} = {val};")
            else:
                print(f"    float next_{prefix}_{i}_{j} = {val};")
                
    print(f"    // updating {prefix} in memory")
    for i in range(expr_matrix.rows):
        for j in range(expr_matrix.cols):
            if expr_matrix.cols == 1:
                print(f"    {prefix}[{i}] = next_{prefix}_{i};")
            else:
                print(f"    {prefix}[{i}][{j}] = next_{prefix}_{i}_{j};")

print("--- Kalman Filter Update Steps in C++ ---")

--- Kalman Filter Update Steps in C++ ---


In [51]:
print("void TinyLKF::predict(float dt, const float u[]) {")

# eq: x_next = F*x + B*u
x_pred = (F * x) + (B * u) if dim_u > 0 else (F * x)
print_cpp_block(x_pred, "x")

# eq: P_next = F*P*F^T + Q
P_pred = (F * P * F.T) + Q
print_cpp_block(P_pred, "P")

print("}")

void TinyLKF::predict(float dt, const float u[]) {
    // --- calculating x ---
    float next_x_0 = 5.0000000000000002e-5*u[0] + x[0] + 0.01*x[1];
    float next_x_1 = 0.01*u[0] + x[1];
    // updating x in memory
    x[0] = next_x_0;
    x[1] = next_x_1;
    // --- calculating P ---
    float next_P_0_0 = P[0][0] + 0.01*P[0][1] + 0.01*P[1][0] + 0.0001*P[1][1] + Q[0][0];
    float next_P_0_1 = P[0][1] + 0.01*P[1][1] + Q[0][1];
    float next_P_1_0 = P[1][0] + 0.01*P[1][1] + Q[1][0];
    float next_P_1_1 = P[1][1] + Q[1][1];
    // updating P in memory
    P[0][0] = next_P_0_0;
    P[0][1] = next_P_0_1;
    P[1][0] = next_P_1_0;
    P[1][1] = next_P_1_1;
}


In [52]:
print("void TinyLKF::update(const float z[]) {")

# equations for the Kalman Filter update step
y = z - (H * x)                 # innovation (residual)
S = (H * P * H.T) + R           # covariance of the innovation
K = P * H.T * S.inv()           # Kalman gain (inversion occurs here!)

# state and covariance update
x_upd = x + (K * y)             
print_cpp_block(x_upd, "x")

P_upd = (I - (K * H)) * P       
print_cpp_block(P_upd, "P")

print("}")  

void TinyLKF::update(const float z[]) {
    // --- calculating x ---
    float next_x_0 = (P[0][0]*z[0] + R[0][0]*x[0])/(P[0][0] + R[0][0]);
    float next_x_1 = (-P[1][0]*(x[0] - z[0]) + x[1]*(P[0][0] + R[0][0]))/(P[0][0] + R[0][0]);
    // updating x in memory
    x[0] = next_x_0;
    x[1] = next_x_1;
    // --- calculating P ---
    float next_P_0_0 = P[0][0]*R[0][0]/(P[0][0] + R[0][0]);
    float next_P_0_1 = P[0][1]*R[0][0]/(P[0][0] + R[0][0]);
    float next_P_1_0 = P[1][0]*R[0][0]/(P[0][0] + R[0][0]);
    float next_P_1_1 = (-P[0][1]*P[1][0] + P[1][1]*(P[0][0] + R[0][0]))/(P[0][0] + R[0][0]);
    // updating P in memory
    P[0][0] = next_P_0_0;
    P[0][1] = next_P_0_1;
    P[1][0] = next_P_1_0;
    P[1][1] = next_P_1_1;
}


In [53]:


header_filename = "TinyLKF.h"

try:
    with open(header_filename, "r") as file:
        h_content = file.read()

    # search for the old #define lines and replace them with the new values
    h_content = re.sub(r"#define\s+LKF_STATE_DIM\s+\d+", f"#define LKF_STATE_DIM {dim_x}", h_content)
    h_content = re.sub(r"#define\s+LKF_MEASURE_DIM\s+\d+", f"#define LKF_MEASURE_DIM {dim_z}", h_content)
    h_content = re.sub(r"#define\s+LKF_CONTROL_DIM\s+\d+", f"#define LKF_CONTROL_DIM {dim_u}", h_content)

    # save the updated content back to the file (overwrites the old file)
    with open(header_filename, "w") as file:
        file.write(h_content)
    print(f"✅ file '{header_filename}' update as sucessfully!")

except FileNotFoundError:
    print(f"⚠️ file '{header_filename}' not found!")


cpp_filename = "TinyLKF.cpp"

# function adapted to return the string instead of printing
def generate_cpp_block(expr_matrix, prefix):
    code = f"    // --- calculating {prefix} ---\n"
    for i in range(expr_matrix.rows):
        for j in range(expr_matrix.cols):
            val = sp.ccode(sp.simplify(expr_matrix[i, j]))
            if expr_matrix.cols == 1:
                code += f"    float next_{prefix}_{i} = {val};\n"
            else:
                code += f"    float next_{prefix}_{i}_{j} = {val};\n"
                
    code += f"    // updating {prefix} in memory\n"
    for i in range(expr_matrix.rows):
        for j in range(expr_matrix.cols):
            if expr_matrix.cols == 1:
                code += f"    {prefix}[{i}] = next_{prefix}_{i};\n"
            else:
                code += f"    {prefix}[{i}][{j}] = next_{prefix}_{i}_{j};\n"
    return code

# building the full C++ code as a string
cpp_code = "#include \"TinyLKF.h\"\n\n"

# --- generating Predict ---
cpp_code += "void TinyLKF::predict(const float u[]) {"
x_pred = (F * x) + (B * u) if dim_u > 0 else (F * x)
cpp_code += generate_cpp_block(x_pred, "x")

P_pred = (F * P * F.T) + Q
cpp_code += generate_cpp_block(P_pred, "P")
cpp_code += "}\n"

# --- generating Update ---
cpp_code += "void TinyLKF::update(const float z[]) {"
y = z - (H * x)                 
S = (H * P * H.T) + R           
K = P * H.T * S.inv()           

x_upd = x + (K * y)             
cpp_code += generate_cpp_block(x_upd, "x")

P_upd = (I - (K * H)) * P       
cpp_code += generate_cpp_block(P_upd, "P")
cpp_code += "}\n"

# writing to the file (overwrites the old one)
with open(cpp_filename, "w") as file:
    file.write(cpp_code)

print(f"✅ file '{cpp_filename}' generated and saved successfully!")

✅ file 'TinyLKF.h' update as sucessfully!
✅ file 'TinyLKF.cpp' generated and saved successfully!
